# Sampler diagnostics: covtype

MCMC against SMC under the MH, DA and HINTS proposals, per iteration and per
second of wallclock.

## Generating the data

The experiments are defined in `examples/incremental_decision_tree/experiments.py` -- three covtype
entries, each spending the same 500,000 proposal draws:

| entry | what it runs | budget |
|---|---|---|
| `covtype_mcmc` | 50 chains x 10,000 iterations | 500,000 |
| `covtype_smc` | 500 particles x 1,000 steps | 500,000 per run |
| `covtype_smc_50p` | 50 particles x 10,000 steps | 500,000 per run |

Run them from the repo root:

```bash
python examples/incremental_decision_tree/run_experiments.py --list           # see them
python examples/incremental_decision_tree/run_experiments.py --index 0 --run-id covtype   # MCMC, ~25 min
python examples/incremental_decision_tree/run_experiments.py --run-id covtype             # all three, hours
```

The MCMC entry alone is enough to make this notebook show something; the SMC
entries take hours (SMC-HINTS is ~4.7 h per run on covtype). Anything not yet
run is skipped below rather than erroring.

Then set `RUN_ID` in the next cell if it is not the newest run.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

# Work from the repo root whether this notebook was opened there or from
# examples/incremental_decision_tree/, so the relative paths below resolve
# either way.
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "discretesampling")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("open this notebook from inside the repository")
    ROOT = parent
os.chdir(ROOT)
sys.path[:0] = [ROOT, os.path.join(ROOT, "examples", "incremental_decision_tree")]

# find_runs_with_metrics, not find_runs: the .h5 files hold the sampler's
# own columns and the stored states, and the predictive metrics live in the
# metrics_*.npz beside them. This folds the two back into one dict per run,
# so `c['test_accuracy']` and `c['metric_iter']` read as they always did.
# Run `python examples/incremental_decision_tree/evaluate_results.py --run-id <id>` first.
from plot_diagnostics import find_runs_with_metrics, known_names  # noqa: E402
from results_io import latest_run_id, read_run_config        # noqa: E402
from discretesampling.domain.incremental_decision_tree import diagnostics as dg  # noqa: E402

RESULTS_ROOT = "Results"
RUN_ID = None          # None -> whichever run is newest

# The experiments to load, and what to call them in the plots. MCMC and SMC
# are separate experiments (they are configured differently to spend the same
# budget), so they are merged here rather than coming out of one file.
EXPERIMENTS = {
    "mcmc": "covtype_mcmc",
    "smc 500p": "covtype_smc",
    "smc 50p": "covtype_smc_50p",
}

run_id = RUN_ID or latest_run_id(RESULTS_ROOT)
if run_id is None:
    raise RuntimeError(f"no runs under {ROOT}/{RESULTS_ROOT} -- "
                       f"see the cell above for the commands that make one")
results_dir = os.path.join(RESULTS_ROOT, run_id)
cfg = read_run_config(results_dir)

runs = {}     # (label, proposal) -> [one dict per chain/run]
kind = {}     # label -> "mcmc" | "smc", for the things that differ between them
for label, name in EXPERIMENTS.items():
    found = find_runs_with_metrics(results_dir, name)
    if not found:
        print(f"[skip] {name!r}: not in this run")
        continue
    for (sampler, proposal), chains in found.items():
        runs[(label, proposal)] = chains
        kind[label] = sampler

if not runs:
    raise RuntimeError(f"none of {list(EXPERIMENTS.values())} are in "
                       f"{results_dir} (it has: "
                       f"{', '.join(known_names(cfg)) or 'nothing'})")

LABELS = [lab for lab in EXPERIMENTS if lab in kind]
print(f"run {run_id!r}")
for (label, proposal), chains in sorted(runs.items()):
    n = len(chains[0]["cumulative_time"])
    print(f"  {label:9s} {proposal:6s}  {len(chains)} run(s), {n} "
          f"{'iterations' if kind[label] == 'mcmc' else 'steps'}, "
          f"{np.mean([c['cumulative_time'][-1] for c in chains]):8.1f}s each")

## Plot helpers

`band` is the idiom every plot uses: the mean over chains/runs, with the
interquartile range across them shaded and the individual runs drawn faintly
underneath.

For the walltime axis the x values differ per run (runs go at slightly
different speeds), so the x plotted is the mean cumulative time across runs at
each recorded point.

In [ ]:
PROPOSALS = ["MH", "DA", "HINTS"]
COLOR = dict(zip(PROPOSALS, ["#4c72b0", "#dd8452", "#55a868"]))


def keys_of(label):
    """This label's (label, proposal) keys, in MH -> DA -> HINTS order."""
    return [(label, p) for p in PROPOSALS if (label, p) in runs]


def series(chains, metric, axis):
    """
    (x, y) for one (label, proposal): y is the metric per run, x is either the
    iteration/step it was recorded at or the cumulative seconds up to it.

    Metrics are recorded on a thinned subset of iterations, so the timing
    columns -- recorded every iteration -- are indexed by `metric_iter` to
    line the two up.
    """
    y = np.vstack([c[metric] for c in chains])
    if axis == "iteration":
        x = np.vstack([c["metric_iter"] for c in chains])
    else:
        x = np.vstack([c["cumulative_time"][c["metric_iter"]] for c in chains])
    return x, y


def band(ax, x, y, color, label, show_runs=True):
    if show_runs and len(y) > 1:
        for xi, yi in zip(x, y):
            ax.plot(xi, yi, color=color, lw=0.5, alpha=0.2)
    lo, hi = np.percentile(y, [25, 75], axis=0)
    ax.fill_between(x.mean(axis=0), lo, hi, color=color, alpha=0.18)
    # Markers only when the series is short enough to read them: a heavily
    # thinned run can be a handful of points, where a bare line is easy to
    # mistake for no data at all, but a dense evaluation gives thousands.
    style = "-o" if y.shape[1] <= 60 else "-"
    ax.plot(x.mean(axis=0), y.mean(axis=0), style, color=color, lw=1.8,
            ms=3, label=label)


def compare(metric, axis, ax, label):
    for key in keys_of(label):
        x, y = series(runs[key], metric, axis)
        band(ax, x, y, COLOR[key[1]], key[1])
    unit = "iteration" if kind[label] == "mcmc" else "step"
    ax.set_xlabel(unit if axis == "iteration" else "cumulative time (s)")
    ax.set_ylabel(metric)
    ax.set_title(f"{label} - {metric} vs {axis}")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)


print("labels:", LABELS)

## Test accuracy: per iteration and per second

The left column is the usual view. The right column is the one that decides
which proposal is worth using: HINTS does several moves' work inside one
iteration, so it should climb faster per *iteration* -- the question is
whether it still does per *second*.

In [ ]:
metric = "test_accuracy"
fig, axes = plt.subplots(len(LABELS), 2, figsize=(13, 4.5 * len(LABELS)),
                         squeeze=False)
for row, label in enumerate(LABELS):
    compare(metric, "iteration", axes[row][0], label)
    compare(metric, "walltime", axes[row][1], label)
fig.tight_layout()

### Everything on one pair of axes

The cross-sampler comparison the budget matching was for: same 500,000
proposal draws on each line, so the walltime panel says which way of spending
them buys the most accuracy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
STYLE = {"mcmc": "-", "smc": "--"}

for axis, ax in zip(("iteration", "walltime"), axes):
    for label in LABELS:
        for key in keys_of(label):
            x, y = series(runs[key], "test_accuracy", axis)
            ax.plot(x.mean(axis=0), y.mean(axis=0), STYLE[kind[label]],
                    color=COLOR[key[1]], lw=1.6,
                    marker="o" if y.shape[1] <= 60 else None, ms=3,
                    label=f"{label} {key[1]}")
    ax.set_xlabel("iteration / step" if axis == "iteration"
                  else "cumulative time (s)")
    ax.set_ylabel("test accuracy")
    ax.set_title(f"all samplers - test accuracy vs {axis}")
    ax.grid(alpha=0.3)
axes[1].set_xscale("symlog")
axes[0].legend(fontsize=8, ncol=2)
fig.tight_layout()

## Every metric, both axes

Accuracy alone hides calibration: a sampler can be right and badly unsure.
Log loss and Brier read the whole predictive distribution, so they separate
the two.

In [ ]:
METRICS = ["test_accuracy", "test_macro_f1", "test_log_loss", "test_brier"]

for label in LABELS:
    fig, axes = plt.subplots(len(METRICS), 2, figsize=(13, 3.4 * len(METRICS)),
                             squeeze=False)
    for row, m in enumerate(METRICS):
        compare(m, "iteration", axes[row][0], label)
        compare(m, "walltime", axes[row][1], label)
    fig.suptitle(label, y=1.001, fontsize=14)
    fig.tight_layout()

## Train vs test

The gap between them is the overfitting the tree prior is meant to hold back.

In [ ]:
fig, axes = plt.subplots(1, len(LABELS), figsize=(6.5 * len(LABELS), 4.5),
                         squeeze=False)
for col, label in enumerate(LABELS):
    ax = axes[0][col]
    for key in keys_of(label):
        for split, style in (("train_accuracy", "--"), ("test_accuracy", "-")):
            x, y = series(runs[key], split, "iteration")
            ax.plot(x.mean(axis=0), y.mean(axis=0), style, color=COLOR[key[1]],
                    lw=1.5, label=f"{key[1]} {split.split('_')[0]}")
    ax.set_xlabel("iteration / step")
    ax.set_ylabel("accuracy")
    ax.set_title(f"{label} - train (dashed) vs test (solid)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()

## Cost per iteration

What the walltime axis is made of. A flat line means cost does not grow with
the tree; a rising one means it does.

In [ ]:
fig, axes = plt.subplots(1, len(LABELS) + 1,
                         figsize=(5.5 * (len(LABELS) + 1), 4.5), squeeze=False)

for col, label in enumerate(LABELS):
    ax = axes[0][col]
    for key in keys_of(label):
        y = np.vstack([c["iter_time"] * 1e3 for c in runs[key]]).mean(axis=0)
        # Rolling mean: per-iteration times are far too noisy to read raw.
        w = max(1, len(y) // 200)
        smooth = np.convolve(y, np.ones(w) / w, mode="valid")
        ax.plot(np.arange(len(smooth)), smooth, color=COLOR[key[1]],
                label=key[1])
    ax.set_xlabel("iteration / step")
    ax.set_ylabel("ms per iteration")
    ax.set_title(f"{label} - cost per iteration")
    ax.set_yscale("log")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

ax = axes[0][-1]
labels, heights, colors = [], [], []
for label in LABELS:
    for key in keys_of(label):
        chains = runs[key]
        total = np.mean([c["cumulative_time"][-1] for c in chains])
        labels.append(f"{label}\n{key[1]}")
        heights.append(total / len(chains[0]["iter_time"]) * 1e3)
        colors.append(COLOR[key[1]])
ax.bar(range(len(labels)), heights, color=colors)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("mean ms per iteration")
ax.set_yscale("log")
ax.set_title("mean cost per iteration")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

## Tree size, acceptance and ESS

The ESS panel is the one to read sceptically: SMC's particles are coupled by
resampling, so an ESS well below the particle count means the ensemble holds
far fewer distinct trees than it appears to.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

ax = axes[0]
for label in LABELS:
    col = "n_nodes" if kind[label] == "mcmc" else "mean_nodes"
    for key in keys_of(label):
        y = np.vstack([c[col] for c in runs[key]]).mean(axis=0)
        ax.plot(np.arange(len(y)), y, color=COLOR[key[1]],
                ls="-" if kind[label] == "mcmc" else ":",
                label=f"{label} {key[1]}")
ax.set_xlabel("iteration / step")
ax.set_ylabel("tree size (nodes)")
ax.set_title("tree size")
ax.grid(alpha=0.3)
ax.legend(fontsize=7)

ax = axes[1]
bars = [(f"{lab}\n{p}", np.mean([c["acceptance_rate"] for c in runs[(lab, p)]]),
         COLOR[p])
        for lab in LABELS if kind[lab] == "mcmc" for (_, p) in keys_of(lab)]
if bars:
    names, vals, cols = zip(*bars)
    ax.bar(range(len(names)), vals, color=cols)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=8)
    ax.set_ylabel("acceptance rate")
    ax.set_title("MCMC acceptance rate")
    ax.grid(alpha=0.3, axis="y")
else:
    ax.set_visible(False)

ax = axes[2]
plotted = False
for label in LABELS:
    if kind[label] != "smc":
        continue
    for key in keys_of(label):
        y = np.vstack([c["ess_history"] for c in runs[key]]).mean(axis=0)
        n_particles = int(runs[key][0]["ess"].max())
        ax.plot(np.arange(len(y)), y / n_particles, color=COLOR[key[1]],
                ls="-" if "500" in label else ":",
                label=f"{label} {key[1]} (N={n_particles})")
        plotted = True
if plotted:
    ax.set_xlabel("step")
    ax.set_ylabel("ESS / particles")
    ax.set_title("SMC effective sample size, as a fraction of N")
    ax.set_yscale("log")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
else:
    ax.set_visible(False)
fig.tight_layout()

## Where the moves went

One row per move considered -- MH considers one per iteration, DA one per
iteration plus a screen, HINTS one per block. `applied` (HINTS) and `proposed`
(MH/DA) are the moves that survived the proposal; whether the sampler then
took them is the acceptance rate above.

In [ ]:
for label in LABELS:
    keys = [k for k in keys_of(label) if "move" in runs[k][0]]
    if not keys:
        continue
    fig, axes = plt.subplots(1, len(keys), figsize=(4.6 * len(keys), 4.2),
                             squeeze=False)
    colors = plt.get_cmap("tab10")(np.linspace(0, 1, len(dg.OUTCOMES)))
    move_names = [m for m in dg.MOVES if m != "stay"]

    for ax, key in zip(axes[0], keys):
        move = np.concatenate([c["move"] for c in runs[key]])
        outcome = np.concatenate([c["outcome"] for c in runs[key]])
        table = np.bincount(move.astype(np.int64) * len(dg.OUTCOMES)
                            + outcome.astype(np.int64),
                            minlength=len(dg.MOVES) * len(dg.OUTCOMES)
                            ).reshape(len(dg.MOVES), len(dg.OUTCOMES))
        bottoms = np.zeros(len(move_names))
        xs = np.arange(len(move_names))
        for oi, oname in enumerate(dg.OUTCOMES):
            h = np.array([table[dg.MOVE_CODE[m], oi] for m in move_names])
            if h.sum():
                ax.bar(xs, h, bottom=bottoms, color=colors[oi], label=oname)
                bottoms += h
        ax.set_xticks(xs)
        ax.set_xticklabels(move_names)
        ax.set_title(f"{label} - {key[1]}")
        ax.set_ylabel("moves considered")
    axes[0][-1].legend(fontsize=7, loc="upper right")
    fig.tight_layout()

## Summary table

Final accuracy against what it cost to get there.

In [ ]:
import pandas as pd

rows = []
for label in LABELS:
    for key in keys_of(label):
        chains = runs[key]
        total = np.mean([c["cumulative_time"][-1] for c in chains])
        n = len(chains[0]["iter_time"])
        row = {
            "experiment": label,
            "proposal": key[1],
            "runs": len(chains),
            "final test acc": np.mean([c["test_accuracy"][-1] for c in chains]),
            "final log loss": np.mean([c["test_log_loss"][-1] for c in chains]),
            "total time (s)": total,
            "ms / iter": total / n * 1e3,
            "full evals": np.mean([c["target_full_evals"] for c in chains]),
            "subset evals": np.mean([c["target_subset_evals"] for c in chains]),
        }
        if kind[label] == "mcmc":
            row["acceptance"] = np.mean([c["acceptance_rate"] for c in chains])
        else:
            row["final ESS"] = np.mean([c["ess_history"][-1] for c in chains])
        rows.append(row)

pd.DataFrame(rows).set_index(["experiment", "proposal"]).round(4)